# KV-cache benchmark: upload both ZIP bundles

This notebook works in a fresh Colab GPU runtime. Keep these two files
on your computer:

1. `colab_kv_cache_bundle.zip` — decoding code, dependencies and development prompts.
2. `finetuning_results.zip` — your completed training results and selected adapter.

Select **Runtime > Change runtime type > GPU**, then run the four steps
below in order. Step 1 asks for the code ZIP. Step 3 asks for the fine-tuning
ZIP, extracts the adapter and starts the benchmark. No files from an earlier
Colab session or pre-existing training folder on Drive are required.

Drive is used to save new benchmark outputs. The experiment reuses your
trained adapter and compares cached/uncached greedy generation with an HF
reference: nine stress cases, 54 timed trials and three full clinical prompts.
Model weights may download into this runtime. Held-out test examples are excluded.


## 1. Upload the KV-cache code bundle

Select **only `colab_kv_cache_bundle.zip`** in this upload dialog.
This cell verifies its contents and extracts the project automatically.


In [ ]:
import hashlib, io, json, zipfile
from pathlib import Path
import torch
from google.colab import files

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")
print("GPU:", torch.cuda.get_device_name(0))
print("Upload colab_kv_cache_bundle.zip")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one code bundle.")
PROJECT = Path("/content/trialgpt_kv_cache")
expected = set(('decoding.py', 'benchmark_kv_cache.py', 'kv_benchmark_report.py', 'test_decoding.py', 'test_kv_benchmark.py', 'KV_CACHE.md', 'baseline.py', 'prepare_model_inputs.py', 'eda.py', 'requirements-colab-finetuning.txt', 'requirements-colab-baseline.txt', 'requirements-model-inputs.txt', 'requirements-data.txt', 'configs/model_input_config.json', 'configs/tokenizer_lock.json', 'configs/kv_cache_config.json', 'data/splits/split_manifest.json', 'data/model_ready/validation.jsonl'))
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    if len(archive.namelist()) != len(expected) + 1 or set(archive.namelist()) != expected | {"bundle_manifest.json"}:
        raise ValueError("Unexpected bundle contents.")
    manifest = json.loads(archive.read("bundle_manifest.json"))
    if set(manifest["file_sha256"]) != expected:
        raise ValueError("Unexpected bundle manifest.")
    for name in expected:
        if hashlib.sha256(archive.read(name)).hexdigest() != manifest["file_sha256"][name]:
            raise ValueError(f"Checksum mismatch: {name}")
    PROJECT.mkdir(parents=True, exist_ok=True)
    archive.extractall(PROJECT)
print("Code bundle verified; held-out test content:", manifest["contains_test_content"])


## 2. Install dependencies and check the decoder

Let installation and all decoder tests finish before continuing.
These miniature CPU checks do not measure full-model GPU performance.


In [ ]:
import subprocess, sys

def run_checked(arguments):
    with subprocess.Popen(arguments, cwd=PROJECT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f"Command failed ({code}); full output is above: {arguments}")

run_checked([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
run_checked([sys.executable, "-m", "pip", "install", "--quiet", "-r",
             str(PROJECT / "requirements-colab-finetuning.txt")])
run_checked([sys.executable, "-m", "unittest", "-v", "test_decoding", "test_kv_benchmark"])


## 3. Upload the fine-tuning results and run the benchmark

Select **only `finetuning_results.zip`** in this upload dialog.
Authorize Drive access if requested. The cell extracts the selected adapter,
verifies preparation and runs the benchmark with that adapter.

New results are saved under `MyDrive/TrialGPT_Assignment2/kv_cache_v1`.
Let the cell finish. If it reports a mismatch or another failure, the last
cell can still download the available diagnostics. An existing run with
different settings requires a different `RUN` folder.


In [ ]:
# Copy this entire file into ONE Colab cell after notebook setup steps 1 and 2.
# This replaces steps 3 and 4: upload results, restore adapter, run benchmark.

import io
import subprocess
import sys
import zipfile
from pathlib import Path

import torch
from google.colab import drive, files

PROJECT = Path("/content/trialgpt_kv_cache")
RUN = Path("/content/drive/MyDrive/TrialGPT_Assignment2/kv_cache_v1")
TRAINING = Path("/content/restored_finetuning")

if not (PROJECT / "benchmark_kv_cache.py").is_file():
    raise FileNotFoundError("Run notebook setup steps 1 and 2 first to load the code bundle and dependencies.")
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU before continuing.")

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

print("Select finetuning_results.zip (the downloaded fine-tuning results).")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one file: finetuning_results.zip.")

required = {
    "training_summary.json",
    "training_manifest.json",
    "adapter/adapter_model.safetensors",
    "adapter/adapter_config.json",
}

with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    if not required.issubset(archive.namelist()):
        raise ValueError("This ZIP does not contain the expected fine-tuning results. Select finetuning_results.zip.")
    destination = TRAINING.resolve()
    for member in archive.infolist():
        if not (destination / member.filename).resolve().is_relative_to(destination):
            raise ValueError("The ZIP contains a path outside the extraction folder.")
    TRAINING.mkdir(parents=True, exist_ok=True)
    archive.extractall(TRAINING)

del uploaded
print("Restored adapter:", TRAINING / "adapter")
print("Benchmark results will be saved to:", RUN)


def run_checked(arguments):
    with subprocess.Popen(
        arguments, cwd=PROJECT, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        exit_code = process.wait()
    if exit_code:
        raise RuntimeError(f"Command failed with exit code {exit_code}. Read the full error above.")


run_checked([
    sys.executable, "-u", "benchmark_kv_cache.py",
    "--training-output", str(TRAINING), "--prepare-only",
])

run_checked([
    sys.executable, "-u", "benchmark_kv_cache.py",
    "--training-output", str(TRAINING), "--output", str(RUN),
])

print((RUN / "summary.md").read_text(encoding="utf-8"))


## 4. Download the results

Run this cell to inspect the summary/plot and download `kv_cache_results.zip`.
Return that ZIP for analysis, including if the benchmark reported a mismatch.


In [ ]:
from IPython.display import Image, display

if (RUN / "summary.md").exists():
    print((RUN / "summary.md").read_text())
if (RUN / "timings.png").exists():
    display(Image(filename=str(RUN / "timings.png")))
if not RUN.exists():
    raise FileNotFoundError("No benchmark output folder exists yet.")
download = Path("/content/kv_cache_results.zip")
with zipfile.ZipFile(download, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in RUN.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(RUN).as_posix())
files.download(str(download))
